# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Paper:** this notebook recomputes every number in the deployed paper (`docs/index.html`). The last code cell writes `work/outputs/capstone_metrics.json` and checks that each headline number printed on the page matches what this notebook computes.

## 1. Question

**Which visible pages keep under-capturing clicks for their search position, and which should an editor review first?**

A page that ranks on page one but earns far fewer clicks than its neighbours at the same position is a cheap opportunity: the visibility already exists, and the snippet (title, meta description) or the intent match is the likely lever. But a one-month dip is often noise. The decision this supports is **the order of a monthly editorial review queue**, not an automatic edit.

**Tension:** the obvious rule, "flag pages with CTR below half of what their position usually gets", flags many pages that recover on their own. Can a model that weighs several signals rank the *lasting* gaps higher, on clients it has never seen?

In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
M = {}   # every number the paper quotes lands here

QUESTION = {
    "question": "Which visible pages keep under-capturing CTR for their position, and which should an editor review first?",
    "unit": "one page (content item) with three consecutive 30-day windows",
    "label": "stays_under: CTR in the NEXT 30 days < 0.5 x pooled CTR of the page's position bucket",
    "decision": "order of a monthly, human-reviewed editorial queue",
    "baseline": "frozen Week-4 rule: LOW_CTR_FOR_POSITION, scored by missed clicks",
}
for k, v in QUESTION.items():
    print(f"{k:>9}: {v}")
print(f"\npython libs: pandas {pd.__version__}, scikit-learn {sklearn.__version__}, seed {SEED}")

 question: Which visible pages keep under-capturing CTR for their position, and which should an editor review first?
     unit: one page (content item) with three consecutive 30-day windows
    label: stays_under: CTR in the NEXT 30 days < 0.5 x pooled CTR of the page's position bucket
 decision: order of a monthly, human-reviewed editorial queue
 baseline: frozen Week-4 rule: LOW_CTR_FOR_POSITION, scored by missed clicks

python libs: pandas 3.0.5, scikit-learn 1.9.1, seed 42


## 2. Data

**Two sources, used for different jobs:**

| | Starter slice (main results) | Warehouse release (replication check, Week 3) |
|---|---|---|
| What | `data/raw/content_refresh_anonymized.csv`, ships with the repo | FlyRank internship warehouse on Hugging Face, build `flyrank_pseudonymized_warehouse_release_v20260703` |
| Size | 30,000 pages × 44 columns, 32 clients | ~79M daily rows in total; I read one partition, `fact_content_daily_performance`, `month=2026-03` (9.8M rows) |
| Windows | three consecutive 30-day windows: days 61–90, 31–60 and the last 30 days before export | Mar 1–15 (features) → Mar 16–31 (label) |
| Grain | one page | one page per day, aggregated to one page per half-month |

The starter slice gives 90-day totals plus the previous and last 30 days. The earliest window is `90d − prev_30d − last_30d`; I check it's never negative.

**Exclusions and why:**
- Positions outside 1–20: too few clicks to compare CTR.
- < 1,000 impressions in 90 days, or < 100 in either modeled 30-day window: CTR on a handful of views is noise.
- Columns that contain the label window or are computed from it: all `*_last_30d` as features, 90-day totals, `trend_pct` / `trend_direction`, `days_since_last_update`.
- Pseudonymous `client_id` / `content_id`: used only to group the validation split, never as features, never shown.
- Nothing public-unsafe enters the paper: no titles, domains, URLs or queries, only aggregates.

In [2]:
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
for m in ["clicks", "impressions", "sessions"]:
    df[f"{m}_early_30d"] = df[f"{m}_90d"] - df[f"{m}_prev_30d"] - df[f"{m}_last_30d"]
    assert (df[f"{m}_early_30d"] >= 0).all(), m

steps = [("starter slice", df)]
s = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20)]; steps.append(("positions 1-20", s))
s = s[s["impressions_90d"] >= 1000]; steps.append((">= 1,000 impressions in 90 days", s))
s = s[(s["impressions_prev_30d"] >= 100) & (s["impressions_last_30d"] >= 100)]; steps.append((">= 100 impressions in each modeled window", s))
funnel = pd.DataFrame([(n, len(x), x["client_id"].nunique()) for n, x in steps], columns=["step", "pages", "clients"])
print(funnel.to_string(index=False))

base = s.copy()
base["bucket"] = pd.cut(base["avg_position"], [0, 3, 5, 10, 15, 20], labels=["1-3", "3-5", "5-10", "10-15", "15-20"])
base = base.reset_index(drop=True)
M["data"] = {"starter_rows": int(len(df)), "starter_cols": int(df.shape[1] - 3), "starter_clients": int(df["client_id"].nunique()),
             "model_pages": int(len(base)), "model_clients": int(base["client_id"].nunique()),
             "funnel": funnel.to_dict(orient="records"),
             # Week-3 warehouse check (executed in Colab against the gated release; numbers copied from w03 outputs)
             "warehouse_check": {"partition_rows": 9_841_378, "lane_pages": 57_439, "lane_clients": 36,
                                 "logreg_auc": 0.758, "logreg_p50": 0.76, "base_rate": 0.504, "leaked_auc": 1.0}}

                                     step  pages  clients
                            starter slice  30000       32
                           positions 1-20  20256       31
          >= 1,000 impressions in 90 days  10011       28
>= 100 impressions in each modeled window   9664       28


## 3. Methodology

**Label (observed, later window).** `stays_under = 1` when the page's CTR in the next 30 days is below half of the pooled CTR of its narrow position bucket (1–3, 3–5, 5–10, 10–15, 15–20) in that same window. Narrow buckets, because CTR was not monotonic across the coarse tiers in this data (Week 2).

**Features (feature window + static context only).** Log impressions, log clicks, log sessions, CTR, CTR ÷ bucket CTR, average position, content age, word count (+ a missing flag), intent and content type.

**Baseline (frozen in Week 4, never tuned on the test).** Flag if CTR < 0.5 × bucket CTR; score = missed clicks at bucket CTR; reason code `LOW_CTR_FOR_POSITION`.

**Models.** Logistic regression (readable), random forest (300 trees, min leaf 20), shallow gradient boosting (depth 3), and the playbook ranking *random forest probability × missed clicks*.

**Validation.** 5-fold GroupKFold by client (test clients never seen in training), and a stricter **time-forward + grouped** design: train on days 61–90 → 31–60, test on days 31–60 → last 30 days, on unseen clients. Metrics: ROC-AUC, precision@20/@50 with the base rate beside them, and the share of all missed clicks captured by the top 50.

**Assumptions.** Pooled bucket CTR is a fair "expected CTR" within this portfolio; 100 impressions is enough to estimate CTR; the 90-day `avg_position` is close enough to the window's position (tested below).

**Leakage checks.** A timeline audit, a harness test that deliberately adds leaky columns (the score must jump), and removal of the one overlapping feature.

In [3]:
def add_window(d, w):
    d[f"ctr_{w}"] = 100 * d[f"clicks_{w}_30d"] / d[f"impressions_{w}_30d"]
    t = d.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    d[f"exp_{w}"] = d["bucket"].map(100 * t[f"clicks_{w}_30d"] / t[f"impressions_{w}_30d"]).astype(float)
    d[f"missed_{w}"] = (d[f"exp_{w}"] / 100 * d[f"impressions_{w}_30d"] - d[f"clicks_{w}_30d"]).clip(lower=0)
    d[f"under_{w}"] = (d[f"ctr_{w}"] < 0.5 * d[f"exp_{w}"]).astype(int)
    return d

def features(d, w):
    X = pd.DataFrame({
        "log_imp": np.log1p(d[f"impressions_{w}_30d"]), "log_clk": np.log1p(d[f"clicks_{w}_30d"]),
        "ctr": d[f"ctr_{w}"], "ratio": d[f"ctr_{w}"] / d[f"exp_{w}"], "avg_position": d["avg_position"],
        "log_sess": np.log1p(d[f"sessions_{w}_30d"]), "content_age_days": d["content_age_days"],
        "has_word_count": d["word_count"].notna().astype(int), "word_count": d["word_count"].fillna(0)}, index=d.index)
    return pd.concat([X, pd.get_dummies(d[["main_intent", "content_type"]], dummy_na=True, dtype=int)], axis=1)

for w in ["prev", "last"]:
    base = add_window(base, w)
X, y = features(base, "prev"), base["under_last"]
rule = np.where(base["under_prev"] == 1, base["missed_prev"], 0.0)
potential = base["missed_prev"].values
gap_next = np.where(y == 1, base["missed_last"], 0.0)
groups = list(GroupKFold(5).split(X, y, base["client_id"]))

p_at = lambda yy, s, k: yy[np.argsort(-s, kind="mergesort")[:k]].mean()
gap_at = lambda g, s, k: g[np.argsort(-s, kind="mergesort")[:k]].sum() / g.sum()
make_rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

# Leakage harness (grouped split, random forest)
def rf_cv(Xv):
    r = []
    for tr, te in groups:
        sc = make_rf().fit(Xv.iloc[tr], y.iloc[tr]).predict_proba(Xv.iloc[te])[:, 1]
        r.append((roc_auc_score(y.iloc[te], sc), p_at(y.values[te], sc, 50)))
    return np.array(r).mean(0)
harness = {
    "Honest feature set": X,
    "+ CTR in the label window (leak)": X.assign(leak=100 * base["clicks_last_30d"] / base["impressions_last_30d"]),
    "+ 90-day clicks (contains label window)": X.assign(leak=np.log1p(base["clicks_90d"])),
    "+ trend_pct (overlapping field)": X.assign(leak=base["trend_pct"]),
    "- avg_position (known 90-day overlap)": X.drop(columns=["avg_position"]),
}
leak = pd.DataFrame({k: rf_cv(v) for k, v in harness.items()}, index=["roc_auc", "p@50"]).T
print(f"Label base rate: {y.mean():.1%} | features: {X.shape[1]}")
print("\nLeakage harness (random forest, grouped by client):")
print(leak.round(3).to_string())
M["label_base_rate"] = round(float(y.mean()), 4)
M["leakage"] = leak.round(3).reset_index(names="variant").to_dict(orient="records")

Label base rate: 37.8% | features: 18

Leakage harness (random forest, grouped by client):
                                         roc_auc   p@50
Honest feature set                         0.791  0.804
+ CTR in the label window (leak)           0.997  1.000
+ 90-day clicks (contains label window)    0.860  1.000
+ trend_pct (overlapping field)            0.796  0.792
- avg_position (known 90-day overlap)      0.788  0.800


## 4. Results (vs baseline)

All rows are scored on **the same five client-grouped folds**; base rate is the share of test pages that really stayed under.

In [4]:
MODELS = {
    "Logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random forest": make_rf,
    "Gradient boosting (depth 3)": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEED),
}
rows = []
for fold, (tr, te) in enumerate(groups, 1):
    yt = y.values[te]
    sc = {"Week-4 rule (baseline)": rule[te]}
    for n, mk in MODELS.items():
        sc[n] = mk().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
    sc["Random forest x missed clicks (playbook)"] = sc["Random forest"] * potential[te]
    for n, s_ in sc.items():
        rows.append({"method": n, "fold": fold, "roc_auc": roc_auc_score(yt, s_), "p@20": p_at(yt, s_, 20),
                     "p@50": p_at(yt, s_, 50), "clicks_captured@50": gap_at(gap_next[te], s_, 50), "base_rate": yt.mean()})
res = pd.DataFrame(rows)
order = ["Week-4 rule (baseline)", "Logistic regression", "Random forest", "Gradient boosting (depth 3)", "Random forest x missed clicks (playbook)"]
main = res.groupby("method")[["roc_auc", "p@20", "p@50", "clicks_captured@50", "base_rate"]].mean().reindex(order)
main["p@50 min"] = res.groupby("method")["p@50"].min().reindex(order)
main["p@50 max"] = res.groupby("method")["p@50"].max().reindex(order)
print("Grouped by client, 5 folds (means):")
print(main.round(3).to_string())

# Validation designs: random vs grouped vs time-forward (random forest probability vs rule)
def design(Xtr, ytr, Xte, yte, rl, splits):
    r = []
    for tr, te in splits:
        s_ = make_rf().fit(Xtr.iloc[tr], ytr.iloc[tr]).predict_proba(Xte.iloc[te])[:, 1]
        r.append((roc_auc_score(yte.iloc[te], s_), p_at(yte.values[te], s_, 50), p_at(yte.values[te], rl[te], 50), yte.values[te].mean()))
    return np.array(r).mean(0)
tf = base[base["impressions_early_30d"] >= 100].copy().reset_index(drop=True)
tf = add_window(add_window(add_window(tf, "early"), "prev"), "last")
Xa, Xb = features(tf, "early").align(features(tf, "prev"), join="outer", axis=1, fill_value=0)
rule_tf = np.where(tf["under_prev"] == 1, tf["missed_prev"], 0.0)
designs = pd.DataFrame({
    "Random 5-fold": design(X, y, X, y, rule, list(KFold(5, shuffle=True, random_state=SEED).split(X))),
    "Grouped by client": design(X, y, X, y, rule, groups),
    "Time-forward + grouped": design(Xa, tf["under_prev"], Xb, tf["under_last"], rule_tf, list(GroupKFold(5).split(Xb, tf["under_last"], tf["client_id"]))),
}, index=["model_auc", "model_p@50", "rule_p@50", "base_rate"]).T
print("\nSame random forest, three validation designs:")
print(designs.round(3).to_string())
M["main_grouped"] = main.round(3).reset_index().to_dict(orient="records")
M["designs"] = designs.round(3).reset_index(names="design").to_dict(orient="records")
M["time_forward_pages"] = int(len(tf))

Grouped by client, 5 folds (means):
                                          roc_auc  p@20   p@50  clicks_captured@50  base_rate  p@50 min  p@50 max
method                                                                                                           
Week-4 rule (baseline)                      0.703  0.76  0.696               0.385      0.355      0.54      0.86
Logistic regression                         0.772  0.80  0.768               0.075      0.355      0.54      0.96
Random forest                               0.791  0.87  0.804               0.208      0.355      0.62      0.94
Gradient boosting (depth 3)                 0.779  0.86  0.740               0.273      0.355      0.50      0.96
Random forest x missed clicks (playbook)    0.717  0.82  0.704               0.406      0.355      0.54      0.84



Same random forest, three validation designs:
                        model_auc  model_p@50  rule_p@50  base_rate
Random 5-fold               0.807       0.884      0.804      0.378
Grouped by client           0.791       0.804      0.696      0.355
Time-forward + grouped      0.779       0.744      0.696      0.355


## 5. Limitations

- **Small, uneven client base.** 28 clients; one client is about 39% of modeled pages. Fold precision@50 ranges widely, so differences of a few points are noise.
- **Starter slice, not the full warehouse.** The main results come from a 30,000-page slice with monthly windows. The Week-3 warehouse check (one month, half-month windows) points the same way but is a different setup, not a benchmark.
- **Position overlap.** `avg_position` is a 90-day average that overlaps the label window. Removing it barely changes the score, but the bucket assignment still uses it.
- **No query-level data.** The worst errors are pages whose impressions halved or tripled, which suggests a query-mix shift the model can't see.
- **No causal claim.** Nothing here shows that rewriting a title recovers clicks. Recorded page updates are workflow events, not measured edits, and updated pages kept their gaps about as often as others.
- **Expected CTR is portfolio-relative**, not an industry benchmark; SERP features (ads, answer boxes) aren't observed.

In [5]:
per_fold = res.pivot(index="fold", columns="method", values="p@50")[["Week-4 rule (baseline)", "Random forest", "Random forest x missed clicks (playbook)"]]
per_fold["test pages"] = [len(te) for _, te in groups]
per_fold["test clients"] = [base.loc[te, "client_id"].nunique() for _, te in groups]
per_fold["base rate"] = res[res["method"] == order[0]].set_index("fold")["base_rate"]
print("Precision@50 per fold (the spread behind the means):")
print(per_fold.round(2).to_string())
largest_share = base["client_id"].value_counts(normalize=True).iloc[0]
print(f"\nLargest client's share of modeled pages: {largest_share:.0%}")
M["largest_client_share"] = round(float(largest_share), 3)
M["per_fold_p50"] = per_fold.round(3).reset_index().to_dict(orient="records")

Precision@50 per fold (the spread behind the means):
method  Week-4 rule (baseline)  Random forest  Random forest x missed clicks (playbook)  test pages  test clients  base rate
fold                                                                                                                        
1                         0.86           0.94                                      0.84        3785             1       0.45
2                         0.78           0.82                                      0.76        1483             4       0.39
3                         0.54           0.62                                      0.54        1470             4       0.23
4                         0.72           0.82                                      0.82        1463             9       0.41
5                         0.58           0.82                                      0.56        1463            10       0.29

Largest client's share of modeled pages: 39%


## 6. Ranked recommendations

The queue comes from the Week-7 playbook (`w07_action_playbook.ipynb`): random forest probability × missed clicks, plus reason codes and an archetype → action map. Summary for the paper:

1. **Rewrite title / meta first on pages with a gap two months running at positions 1–5.** Visible, consistently losing the click; the snippet is the likely lever.
2. **For persistent gaps at positions 5–20, check intent before polishing the snippet.** Lower visibility; the page may answer a different question than the query.
3. **Aging pages (365+ days) with a persistent gap get a freshness review on top.** Older pages carry more gaps and keep them longer.
4. **Check the query mix before editing pages whose impressions halved or doubled.** These produced the worst model errors.
5. **Re-check one-month gaps next month instead of editing.** About half of them don't repeat.
6. **Never automate:** publishing rewrites, deleting/redirecting pages, client-facing scores, or promised click gains. Every row is human-reviewed.

In [6]:
pb = json.loads((root / "work/outputs/playbook_metrics.json").read_text())
hist = tf.assign(history=np.select([(tf["under_early"] == 1) & (tf["under_prev"] == 1), tf["under_prev"] == 1, tf["under_early"] == 1],
                                   ["gap 2 months running", "gap last month only", "gap 2 months ago only"], "no gap"))
persist = hist.groupby("history")["under_last"].agg(share="mean", pages="size").reindex(
    ["no gap", "gap 2 months ago only", "gap last month only", "gap 2 months running"])
print("Share under-capturing next month, by gap history:")
print(persist.round(3).to_string())
print("\nPlaybook receipt (time-forward, unseen clients):")
for r in pb["receipt_time_forward_grouped"]:
    print(f"  {r['ranking']:<36} p@50 {r['precision@50']:.3f} | clicks captured@50 {r['clicks_gap_captured@50']:.3f}")
print("\nTop-100 actions:", pb["top100_actions"])
print("Top-50 cost/value:", pb["top50"])
M["persistence"] = persist.round(3).reset_index().to_dict(orient="records")
M["playbook"] = {k: pb[k] for k in ["receipt_time_forward_grouped", "top100_actions", "top50", "persistence_by_age", "scored_pages"]}

Share under-capturing next month, by gap history:
                       share  pages
history                            
no gap                 0.145   4319
gap 2 months ago only  0.441   1206
gap last month only    0.499   1400
gap 2 months running   0.662   2700

Playbook receipt (time-forward, unseen clients):
  Frozen Week-4 rule                   p@50 0.696 | clicks captured@50 0.386
  Playbook queue (p x missed clicks)   p@50 0.712 | clicks captured@50 0.419

Top-100 actions: {'REWRITE_TITLE_META + INTENT_CHECK': 38, 'REWRITE_TITLE_META': 24, 'REWRITE_TITLE_META + FRESHNESS_REVIEW': 14, 'CHECK_QUERY_MIX': 13, 'RECHECK_NEXT_MONTH': 8, 'NO_ACTION': 3}
Top-50 cost/value: {'editor_hours_assumed': 22.8, 'clicks_at_stake_30d': 7808, 'random_50_gap_pages_clicks_at_stake': 416, 'expected_persisting_clicks_30d': 5672}


## 7. Artifacts the paper embeds

- `work/outputs/capstone_metrics.json`: every number on the page (committed, the receipts).
- `docs/index.html`: the paper. Its charts are drawn in HTML from these numbers, so they stay sharp on phones.
- The cell below checks that the headline numbers written in the paper match this run. If a rerun changes a number, the check fails and the page must be updated.

In [7]:
out = root / "work/outputs"; out.mkdir(parents=True, exist_ok=True)
(out / "capstone_metrics.json").write_text(json.dumps(M, indent=2, default=str))

f2 = lambda v: f"{v:.2f}"
mg = main.round(3)
expected_on_page = {
    "pages modeled": f"{M['data']['model_pages']:,}",
    "clients modeled": str(M["data"]["model_clients"]),
    "base rate": f"{M['label_base_rate']:.1%}",
    "rule p@50": f2(mg.loc["Week-4 rule (baseline)", "p@50"]),
    "RF p@50": f2(mg.loc["Random forest", "p@50"]),
    "RF AUC": f2(mg.loc["Random forest", "roc_auc"]),
    "rule AUC": f2(mg.loc["Week-4 rule (baseline)", "roc_auc"]),
    "playbook clicks captured": f2(mg.loc["Random forest x missed clicks (playbook)", "clicks_captured@50"]),
    "rule clicks captured": f2(mg.loc["Week-4 rule (baseline)", "clicks_captured@50"]),
    "random-split p@50": f2(designs.loc["Random 5-fold", "model_p@50"]),
    "time-forward p@50": f2(designs.loc["Time-forward + grouped", "model_p@50"]),
    "time-forward AUC": f2(designs.loc["Time-forward + grouped", "model_auc"]),
    "persist 2 months": f"{persist.loc['gap 2 months running', 'share']:.0%}",
    "persist 1 month": f"{persist.loc['gap last month only', 'share']:.0%}",
    "persist none": f"{persist.loc['no gap', 'share']:.0%}",
}
paper = root / "docs/index.html"
html = paper.read_text() if paper.exists() else ""
text = re.sub(r"<[^>]+>", " ", html)
check = pd.DataFrame([(k, v, v in text) for k, v in expected_on_page.items()], columns=["number", "value", "on page"])
print(check.to_string(index=False))
assert check["on page"].all(), "paper and notebook disagree - update docs/index.html"
for banned in ["proves", "caused", "will increase", "guarantee"]:
    assert banned not in text.lower(), banned
print("\nPaper numbers match this run; no banned claim words.")

                  number value  on page
           pages modeled 9,664     True
         clients modeled    28     True
               base rate 37.8%     True
               rule p@50  0.70     True
                 RF p@50  0.80     True
                  RF AUC  0.79     True
                rule AUC  0.70     True
playbook clicks captured  0.41     True
    rule clicks captured  0.39     True
       random-split p@50  0.88     True
       time-forward p@50  0.74     True
        time-forward AUC  0.78     True
        persist 2 months   66%     True
         persist 1 month   50%     True
            persist none   14%     True

Paper numbers match this run; no banned claim words.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 8. Demo outline: 5 minutes for the Week-8 showcase

| Time | Beat | What I say / show |
|---|---|---|
| 0:00–0:45 | **Question** | FlyRank flags "Fix CTR" pages: visible in search, few clicks. An editor has time for ~20 a week. *Which visible pages keep under-capturing CTR for their position, and which should be reviewed first?* |
| 0:45–2:00 | **Method** | Label observed in the *next* month (CTR < half of the position bucket's pooled CTR). Features only from the month before. Frozen baseline rule vs. a random forest. Validation on clients the model never saw, then time-forward. Planted-leak test: adding the label-month CTR sends AUC to 1.00, so the harness works. |
| 2:00–3:15 | **One chart** | "Precision@50 under three validation designs": 0.88 random split → 0.80 grouped by client → 0.74 time-forward, against 0.80 / 0.70 / 0.70 for the rule. The stricter the test, the smaller the lead. |
| 3:15–4:00 | **One honest result** | On unseen clients, one month ahead, the model measured precision@50 of 0.74 vs 0.70 for the rule (base rate 37.8%): a modest, directional lift. Separately, a gap seen two months running persisted 66% of the time vs 50% after one month. Not shown: that rewriting a title recovers clicks. |
| 4:00–4:40 | **One recommendation** | Wait for a gap to repeat, then rewrite title/meta on persistent gaps at positions 1–5 first, ranked by likely-real missed clicks. A human reviews every page; nothing auto-publishes. |
| 4:40–5:00 | **Close** | Next step: hold back ~20% of edit candidates for a month to measure whether edits help. Paper link + repo. |

Backup if asked: fold spread (p@50 0.62–0.94 across client folds), the 90-day position overlap (removing it barely moves the score), and why ranking by probability alone captured fewer missed clicks (0.21 vs 0.39).

## 9. Two shareable cuts

**Social post (methodology)**

> My first ML model looked great: 0.88 precision on a random split. Then I tested it honestly.
>
> The task: on real, pseudonymized FlyRank search data, rank pages that rank well but keep losing clicks for their position, so an editor knows what to fix first.
>
> Three validation designs, same model:
> • random split: 0.88 precision@50
> • unseen clients: 0.80
> • trained on the past, tested on the next month: 0.74
>
> A simple rule scored 0.70 on the strict test. So the real lift is modest, and the stricter the test, the smaller it got.
>
> Two habits I'm keeping: plant a leak on purpose (if the score doesn't jump to ~1.0, your test harness is broken), and always print the base rate next to the metric.
>
> The most useful finding wasn't the model: a CTR gap seen two months running persisted 66% of the time, vs 50% after one month. Wait for the gap to repeat.
>
> Paper and notebooks: https://maksimnikolin.github.io/FlyRank/

**Employer summary (3 sentences)**

> I built a validated, human-reviewed ranking that tells content editors which search-visible pages keep under-capturing clicks for their position, with reason codes and an action for each page. It runs on pseudonymized production search data from the FlyRank ML Internship: a 30,000-page slice across 32 client sites, plus a replication check on one month of the ~79M-row warehouse. Tested on clients and a month it never saw, it measured precision@50 of 0.74 vs 0.70 for a transparent rule (base rate 37.8%), and it showed that gaps lasting two months are the ones worth fixing first (66% persist vs 50%).